# Data Loading, Cleaning & Export

### Importazione Librerie

In [1]:
import numpy as np
import pandas as pd
import requests

### Caricamento Dati

In [50]:
CSV_files = {
    "customers":           r"dataset/olist_customers_dataset.csv",
    "geolocation_dataset": r"dataset/olist_geolocation_dataset.csv",
    "order_items":         r"dataset/olist_order_items_dataset.csv",
    "order_payments":      r"dataset/olist_order_payments_dataset.csv",
    "order_review":        r"dataset/olist_order_reviews_dataset.csv",
    "order_dataset":       r"dataset/olist_orders_dataset.csv",
    "list_product":        r"dataset/olist_products_dataset.csv",
    "list_seller":         r"dataset/olist_sellers_dataset.csv",
    "product_category":    r"dataset/product_category_name_translation.csv"
}

dataframes = {name: pd.read_csv(path) for name, path in CSV_files.items()}

df_customers           = dataframes["customers"]
df_orders_items        = dataframes["order_items"]
df_order_payments      = dataframes["order_payments"]
df_order_review        = dataframes["order_review"]
df_order_dataset       = dataframes["order_dataset"]
df_list_product        = dataframes["list_product"]
df_list_seller         = dataframes["list_seller"]
df_product_category    = dataframes["product_category"]
df_geolocation_dataset = dataframes["geolocation_dataset"]

### Analisi Preliminare

In [3]:
# Stampa un riepilogo di ogni csv con la sua shape, i missing values e i duplicati.
def quick_overview(df, name):

    # Nome DF
    print(f"{name}")
    print(f"Shape: {df.shape}")

    # Valori nulli
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"Valori nulli:\n{nulls.to_string()}\n")
    else:
        print("Nessun valore nullo\n")

for name, df in dataframes.items():
    quick_overview(df, name)

customers
Shape: (99441, 5)
Nessun valore nullo

geolocation_dataset
Shape: (1000163, 5)
Nessun valore nullo

order_items
Shape: (112650, 7)
Nessun valore nullo

order_payments
Shape: (103886, 5)
Nessun valore nullo

order_review
Shape: (99224, 7)
Valori nulli:
review_comment_title      87656
review_comment_message    58247

order_dataset
Shape: (99441, 8)
Valori nulli:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

list_product
Shape: (32951, 9)
Valori nulli:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

list_seller
Shape: (3095, 4)
Nessun valore nullo

product_category
Shape: (71, 2)
Nessun valore nullo



### PULIZIA: df_list_product

In [4]:
# 1) Droppiamo le colonne (peso, misure)
# 2) Droppiamo i 610 prodotti senza categoria (< 2% del totale):
#    verranno esclusi anche dagli altri df.
# 3) Uniamo la traduzione inglese della categoria.

# 1
df_list_product = df_list_product.drop(
    columns=['product_name_lenght', 'product_width_cm',
             'product_height_cm', 'product_length_cm', 'product_weight_g']
)

# 2
# Salviamo gli id dei prodotti senza categoria per filtrare order_items dopo
product_id_nan = df_list_product[df_list_product['product_category_name'].isna()]
product_id_to_delete = product_id_nan['product_id'].to_numpy()
df_list_product = df_list_product.dropna(subset=['product_category_name'])

# 3
df_list_product = (
    df_list_product
    .merge(df_product_category, on='product_category_name', how='left')
)

df_list_product.info()
df_list_product.sample(2)

<class 'pandas.DataFrame'>
RangeIndex: 32341 entries, 0 to 32340
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   product_id                     32341 non-null  str    
 1   product_category_name          32341 non-null  str    
 2   product_description_lenght     32341 non-null  float64
 3   product_photos_qty             32341 non-null  float64
 4   product_category_name_english  32328 non-null  str    
dtypes: float64(2), str(3)
memory usage: 1.2 MB


,product_id,product_category_name,product_description_lenght,product_photos_qty,product_category_name_english
9055,d983df6d3977955580e8585217b9ee42,automotivo,972.0,2.0,auto
8239,0c753fe6a58dfa7a27bda5de76e779c3,esporte_lazer,1127.0,1.0,sports_leisure


In [5]:
# Traduciamo con Selenium le categorie portoghesi senza traduzione inglese
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# Troviamo i nomi portoghesi che non hanno traduzione nel dataframe df_product_category
traduzione_mancante = df_list_product['product_category_name_english'].isna()
nomi_mancanti = df_list_product[traduzione_mancante]['product_category_name'].unique()
print(f"Categorie da tradurre ({len(nomi_mancanti)}): \n{nomi_mancanti}\n")


Categorie da tradurre (2): 
<StringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: str



In [6]:
# Apre Google Translate nel browser e restituisce la traduzione
# dal portoghese all'inglese del testo passato, formattata in snake_case.
def traduci_categoria(testo):

    # Costruisce l'URL di Google Translate con i parametri: lingua portoghese (pt), lingua inglese (en) e testo
    url = f"https://translate.google.com/?sl=pt&tl=en&text={testo}&op=translate"
    driver.get(url)
    
    # Attende 2 secondi che la pagina carichi e la traduzione appaia
    time.sleep(2)

    # Accetta i cookie se il popup è presente
    try:
        accept_button = driver.find_element(By.XPATH, "//button[.//span[text()='Accept all']]")
        accept_button.click()
        time.sleep(2)
    except:
        pass  # Il popup non è presente, continua normalmente
    
    # Trova l'elemento HTML che contiene il testo tradotto
    risultato = driver.find_element(By.XPATH, "//span[@jsname='W297wb']")
    
    # Converte in minuscolo e sostituisce gli spazi con underscore
    traduzione = risultato.text.lower().replace(' ', '_')
    return traduzione

# Avvia Firefox
# NB CONTROLLATE TUTTI SE SI APRE IL BROWSER E VEDETE GOOGLE TRANSLATE
driver = webdriver.Firefox()

# Dizionario che conterrà le traduzioni: {nome_portoghese: traduzione_inglese}
traduzioni_manuali = {}

# Itera sui nomi di categoria mancanti di traduzione
for nome in nomi_mancanti:
    traduzione = traduci_categoria(nome)
    traduzioni_manuali[nome] = traduzione
    print(f"{nome}  →  {traduzione}")
    time.sleep(1)

# Chiude il browser al termine delle traduzioni
driver.quit()

# Aggiorna il DataFrame: per ogni categoria tradotta,
# riempie i valori NaN nella colonna 'product_category_name_english'
for nome, traduzione in traduzioni_manuali.items():
    df_list_product.loc[
        df_list_product['product_category_name'] == nome,
        'product_category_name_english'
    ] = traduzione

df_list_product.info()

NoSuchElementException: Message: Unable to locate element: //span[@jsname='W297wb']; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:555:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:136:16


### PULIZIA: df_orders_items

In [9]:
url = 'https://v6.exchangerate-api.com/v6/ee8e65018f5adf36f58283bd/latest/EUR'
oae = requests.get(url)
print(oae.status_code)#richiesta API  

200


In [10]:
EUR_BRL=oae.json()['conversion_rates']['BRL']#Stanziata la variabile del tasso di cambio chiamndolo EUR_BRL
EUR_BRL#tasso conversione

6.1312

In [11]:
# 1) Escludiamo gli items con product_id in product_id_to_delete
# 2) Convertiamo shipping_limit_date in datetime

# 1
df_orders_items = (
    df_orders_items[~df_orders_items['product_id'].isin(product_id_to_delete)]
    .copy()
    .reset_index(drop=True)
)

# 2
df_orders_items['shipping_limit_date'] = pd.to_datetime(
    df_orders_items['shipping_limit_date']
)

#conversione in euro di price e freight_value
df_orders_items['eur_price']=round(df_orders_items['price']/EUR_BRL,2)
df_orders_items['eur_freight_value']=round(df_orders_items['freight_value']/EUR_BRL,2)
#rimozione colonne in real breasiliano
df_orders_items=df_orders_items.drop(['price','freight_value'],axis=1)


df_orders_items.info()
df_orders_items.sample(2)

<class 'pandas.DataFrame'>
RangeIndex: 111047 entries, 0 to 111046
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             111047 non-null  str           
 1   order_item_id        111047 non-null  int64         
 2   product_id           111047 non-null  str           
 3   seller_id            111047 non-null  str           
 4   shipping_limit_date  111047 non-null  datetime64[us]
 5   eur_price            111047 non-null  float64       
 6   eur_freight_value    111047 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 5.9 MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,eur_price,eur_freight_value
101770,ea733405d742855b5e66606f15b493c7,1,bf818ef79f0bf94fcefe6d0f1c707cda,d20b021d3efdf267a402c402a48ea64b,2018-03-15 18:15:40,17.55,2.30
56929,836e4cb15d850d5cb77cda4c3c387665,1,7926c1689244625162673c66b4196371,ff69aa92bb6b1bf9b8b7a51c2ed9cf8b,2018-03-06 16:15:44,190.50,19.95


### PULIZIA: df_order_dataset

In [12]:
# 1) Conversione delle colonne data in datetime

# 2) delivery_delay_days : differenza tra consegna effettiva e stimata
#    (positivo = in ritardo, negativo = in anticipo)
#    actual_delivery_days: giorni totali dall'acquisto alla consegna

# 1
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df_order_dataset[col] = pd.to_datetime(df_order_dataset[col])

# 2
df_order_dataset['delivery_delay_days'] = (
    df_order_dataset['order_delivered_customer_date'] -
    df_order_dataset['order_estimated_delivery_date']
).dt.days
df_order_dataset['actual_delivery_days'] = (
    df_order_dataset['order_delivered_customer_date'] -
    df_order_dataset['order_purchase_timestamp']
).dt.days

df_order_dataset.info()
df_order_dataset.sample(2)

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   delivery_delay_days            96476 non-null  float64       
 9   actual_delivery_days           96476 non-null  float64       
dtypes: datetime64[us](5), float64(2), str(3)
memory usage: 7.6 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,actual_delivery_days
54926,f2b609cb8300d9cf7f2731582909bd61,4107cf3182f56b9dc9ce104c7379434c,delivered,2017-09-20 16:32:20,2017-09-20 16:45:32,2017-09-22 19:11:56,2017-09-25 21:13:48,2017-10-05,-10.0,5.0
83383,3756637ed2fc243af380469a2aa5384c,7fdc7f83bc78ab2261e57b8f0550a710,delivered,2018-02-05 10:48:41,2018-02-05 11:12:58,2018-02-06 18:24:53,2018-02-07 16:51:58,2018-02-26,-19.0,2.0


### PULIZIA: df_order_review

In [13]:
# 1) Conversione della colonna review_creation_date in datetime
# 2) Droppiamo le colonne (review_comment_title, review_comment_message)

# 1
df_order_review['review_creation_date'] = pd.to_datetime(
    df_order_review['review_creation_date']
)

# 2
df_order_review = df_order_review.drop(columns= ['review_comment_title', 'review_comment_message'])

df_order_review.sample(2)

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
5469,a11650bf37cf1211d7532e7501ab1144,753409eb1dc2fceb6dbb2a350a35ffc8,5,2017-06-20,2017-06-20 23:05:29
4223,fc43b17397bf3bd469100cdfb186d6e5,c6735c93a3773ffe872b4184ccfdad09,5,2018-05-01,2018-05-02 18:30:29


### PULIZIA: df_order_payments

In [14]:
# Per un ordine ci sono diverse righe di pagamenti
# Aggreghiamo per order_id

df_order_payments = (
    df_order_payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value= ('payment_value', 'sum'),
        payment_installments= ('payment_installments', 'max'),
        payment_type         = ('payment_type', 'first')
    )
)

df_order_payments.sample(2)

,order_id,total_payment_value,payment_installments,payment_type
61771,9fce3cd33d8928808f389d86ea7556e6,91.23,1,credit_card
15180,27572d0c11af71f78423d11722b2503b,193.81,4,credit_card


In [15]:
df_order_payments['eur_total_payment_value']=round(df_order_payments['total_payment_value']/EUR_BRL,2)#convesrione
df_order_payments=df_order_payments.drop(['total_payment_value'],axis=1)

In [ ]:
df_order_payments.sample(2)


<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


### PULIZIA:df_geolocation

In [17]:
# Coordinate del Brasile
brazil = {
    "lat_min": -34.0, "lat_max": 5.0,
    "lng_min": -75.0, "lng_max": -28.0
}

# Filtra coordinate fuori dal Brasile
df_geolocation_clean = df_geolocation_dataset[
    df_geolocation_dataset["geolocation_lat"].between(brazil["lat_min"], brazil["lat_max"]) &
    df_geolocation_dataset["geolocation_lng"].between(brazil["lng_min"], brazil["lng_max"])
].copy()

# Raggruppa per zip code (media lat/lng, primo valore per città e stato)
df_geolocation_dataset = (
    df_geolocation_clean
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat   = ('geolocation_lat',   'mean'),
        geolocation_lng   = ('geolocation_lng',   'mean'),
        geolocation_city  = ('geolocation_city',  'first'),
        geolocation_state = ('geolocation_state', 'first')
    )
)

df_geolocation_dataset.info()
df_geolocation_dataset.sample(2)

<class 'pandas.DataFrame'>
RangeIndex: 19011 entries, 0 to 19010
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  19011 non-null  int64  
 1   geolocation_lat              19011 non-null  float64
 2   geolocation_lng              19011 non-null  float64
 3   geolocation_city             19011 non-null  str    
 4   geolocation_state            19011 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 742.7 KB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
13206,64730,-7.659327,-41.881402,campinas do piaui,PI
17111,87120,-23.614374,-52.082794,floresta,PR


### PULIZIA: df_list_seller

In [18]:
df_list_seller=df_list_seller.rename(columns={'seller_zip_code_prefix':'geolocation_zip_code_prefix'})
df_list_seller.sample(3)

,seller_id,geolocation_zip_code_prefix,seller_city,seller_state
2565,b7ed9fb14c8eadb37adb9c45d67ab0fb,12331,jacarei,SP
2336,28f10b1c5e5abb9d4857745bede6147c,4250,sao paulo,SP
2663,4a3ccda38b2129705f3fb522db62ca31,17504,marilia,SP


### PULIZIA: df_customers

In [19]:
df_customers=df_customers.rename(columns={'customer_zip_code_prefix':'geolocation_zip_code_prefix'})
df_customers.sample(3)

,customer_id,customer_unique_id,geolocation_zip_code_prefix,customer_city,customer_state
66694,26470d2cf0c109dcb7cc046d48779846,f5b18005a9e442583066a5527f119388,13215,jundiai,SP
97010,73a77e205798f6f820f6d70908b0be0a,6656efd9580812cee9c6b10e3574ed11,87910,santa isabel do ivai,PR
93626,a87d45ca887d3d335a94649960190869,194372be77b95cc7e9e0d96e6d9bfb07,17032,bauru,SP


### CREAZIONE DI DATAFRAME GLOBAL

#### Dataframe global (aggregazione per ordini)

In [85]:
# 1. Merge fra list_seller e geolocation_clean
df_seller_geo = df_list_seller.merge(
    df_geolocation_clean,
    on='geolocation_zip_code_prefix',
    how='left'
)
#dal merge ci sono 7 valori nulli per lat, lng e state >> rimpiazzo i valori con 0 e missing
df_seller_geo['geolocation_lat'] = df_seller_geo['geolocation_lat'].fillna(0)
df_seller_geo['geolocation_lng'] = df_seller_geo['geolocation_lng'].fillna(0)
df_seller_geo['geolocation_city'] = df_seller_geo['geolocation_city'].fillna('missing')
df_seller_geo['geolocation_state'] = df_seller_geo['geolocation_state'].fillna('missing')

# 2. Merge fra customers e geolocation_clean
df_customer_geo = df_list_seller.merge(
    df_geolocation_clean,
    on='geolocation_zip_code_prefix', 
    how='left'
)
#dal merge ci sono 7 valori nulli per lat, lng e state >> rimpiazzo i valori con 0 e missing
df_customer_geo['geolocation_lat'] = df_customer_geo['geolocation_lat'].fillna(0)
df_customer_geo['geolocation_lng'] = df_customer_geo['geolocation_lng'].fillna(0)
df_customer_geo['geolocation_city'] = df_customer_geo['geolocation_city'].fillna('missing')
df_customer_geo['geolocation_state'] = df_customer_geo['geolocation_state'].fillna('missing')

# 3. Aggregazione order_items per order_id
agg_orders_items = (df_orders_items.groupby('order_id', as_index = False).agg(
    total_items=('order_item_id', 'count'),       # numero di prodotti venduti
    total_price=('eur_price', 'sum'),                 # somma prezzi
    freight_total=('eur_freight_value', 'sum'),       # somma costi spedizione
    freight_avg=('eur_freight_value', 'mean'),        # media costi spedizione
    first_shipping_limit=('shipping_limit_date', 'min')  # primo limite spedizione
)
)

# 4. Aggregazione order_payments per order_id
agg_order_payments = (
    df_order_payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value= ('eur_total_payment_value', 'sum'),
        payment_installments= ('payment_installments', 'max'),
        payment_type         = ('payment_type', 'first')
    )
)

# 5. Aggregazione order_review per order_id
agg_order_review = (
    df_order_review
    .groupby('order_id', as_index=False)
    .agg(
    avg_review=('review_score', 'mean'),
    num_reviews=('review_score', 'count'),
    firs_review_date = ('review_creation_date', 'min'),
    first_review_answer = ('review_answer_timestamp', 'min')
    )
)

# 6. Aggregazione list_product per order_id
agg_list_products = (
    df_orders_items
    .merge(df_list_product, on='product_id', how='left')
    .groupby('order_id', as_index=False)
    .agg(
        num_products=('product_id', 'count'),
        num_categories=('product_category_name_english', 'nunique')
    )
)

# . Aggregazione list_seller per order_id
agg_seller = (
    df_orders_items
    .merge(df_seller_geo, on='seller_id', how='left')
    .groupby('order_id', as_index=False)
    .agg(
        num_sellers = ('seller_id', 'count'),
        geolocation_zip_code_prefix = ('geolocation_zip_code_prefix', 'first'),
        seller_city = ('seller_city', 'first'),
        seller_state = ('seller_state', 'first')
    )   
)        

# Merge
df_global = df_order_dataset.merge(df_customers, on='customer_id', how='left') \
                     .merge(agg_orders_items, on='order_id', how='left') \
                     .merge(agg_order_review, on='order_id', how='left') \
                     .merge(agg_order_payments, on='order_id', how='left') \
                     .merge(agg_list_products, on='order_id', how='left') \
                     .merge(agg_seller, on='order_id', how='left')
                     



KeyError: 'geolocation_zip_code_prefix'

In [86]:
df_global.info()

<class 'pandas.DataFrame'>
Index: 90655 entries, 0 to 99440
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       90655 non-null  str           
 1   customer_id                    90655 non-null  str           
 2   order_status                   90655 non-null  str           
 3   order_purchase_timestamp       90655 non-null  datetime64[us]
 4   order_approved_at              90642 non-null  datetime64[us]
 5   order_delivered_carrier_date   90655 non-null  datetime64[us]
 6   order_delivered_customer_date  90655 non-null  datetime64[us]
 7   order_estimated_delivery_date  90655 non-null  datetime64[us]
 8   delivery_delay_days            90655 non-null  float64       
 9   actual_delivery_days           90655 non-null  float64       
 10  customer_unique_id             90655 non-null  str           
 11  geolocation_zip_code_prefix_x  

In [ ]:
#PULIZIA DATI

# 1768 ordini non hanno review - Inserisco 0 al posto del valore nullo
df_global['avg_review'] = df_global['avg_review'].fillna(0)
df_global['num_reviews'] = df_global['num_reviews'].fillna(0)
# Feature booleana che identifica 0 come valore nullo e non come score
df_global['has_review'] = df_global['num_reviews'] > 0

# Controllo i valori nulli (97277)
cols_items_seller = [
    'total_items', 'total_price', 'freight_total', 'freight_avg',
    'num_products', 'num_categories', 'num_sellers',
    'seller_city', 'seller_state', 'geolocation_zip_code_prefix_y'
]
rows_with_nulls = df_global[df_global[cols_items_seller].isna().any(axis=1)]
same_nulls = rows_with_nulls[cols_items_seller].isna().all(axis=1)

#Sostituisco i valori stringa con "unknown"
cat_cols = ['seller_city', 'seller_state']
df_global[cat_cols] = df_global[cat_cols].fillna('unknown')
df_global['has_items'] = df_global['total_items'] > 0

# Sostituisco null nei valori numerici dei pagamenti
df_global['total_payment_value'] = df_global['total_payment_value'].fillna(0)
df_global['payment_installments'] = df_global['payment_installments'].fillna(0)
# Sostituisco null nel metodo di pagamento con 'unknown'
df_global['payment_type'] = df_global['payment_type'].fillna('unknown')
# Facoltativo: feature booleana se ordine ha pagamento
df_global['has_payment'] = df_global['total_payment_value'] > 0

In [89]:
df_global

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,actual_delivery_days,...,payment_type,num_sellers,geolocation_zip_code_prefix_y,seller_city,seller_state,has_review,has_items,has_payment,approval_time_hours,estimated_delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,8.0,...,credit_card,207.0,9350.0,maua,SP,True,True,True,0.178333,15
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,13.0,...,boleto,71.0,31570.0,belo horizonte,SP,True,True,True,30.713889,19
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,9.0,...,credit_card,170.0,14840.0,guariba,SP,True,True,True,0.276111,26
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,13.0,...,credit_card,95.0,31842.0,belo horizonte,MG,True,True,True,0.298056,26
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,2.0,...,credit_card,47.0,8752.0,mogi das cruzes,SP,True,True,True,1.030556,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,-11.0,8.0,...,credit_card,55.0,12913.0,braganca paulista,SP,True,True,True,0.000000,18
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,-2.0,22.0,...,credit_card,92.0,17602.0,tupa,SP,True,True,True,0.194167,23
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,-6.0,24.0,...,credit_card,325.0,8290.0,sao paulo,SP,True,True,True,0.292500,30
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-21.0,17.0,...,credit_card,100.0,37175.0,ilicinea,MG,True,True,True,0.131667,37


In [25]:
df_global.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 35 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   delivery_delay_days            96476 non-null  float64       
 9   actual_delivery_days           96476 non-null  float64       
 10  customer_unique_id             99441 non-null  str           
 11  geolocation_zip_code_prefi

### EXPORT: tutti i dataframe puliti nella cartella output

In [88]:
import os

# Crea la cartella output se non esiste
os.makedirs("output", exist_ok=True)

dataframes_to_export = {
    "list_product":        df_list_product,
    "orders_items":        df_orders_items,
    "order_dataset":       df_order_dataset,
    "order_review":        df_order_review,
    "order_payments":      df_order_payments,
    "geolocation_dataset": df_geolocation_dataset,
    "list_seller":         df_list_seller,
    "customers":           df_customers,
}

for name, df in dataframes_to_export.items():
    df.to_csv(f"output/{name}.csv", index=False)

### Regressione Lineare Multivariata

In [51]:
df_customers          
     

df_order_review     
          

df_order_payments      
  
order_it=df_orders_items.drop(['price','freight_value','shipping_limit_date','seller_id'],axis=1)
 
df_order_dataset

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [12]:
product=df_list_product.drop(['product_name_lenght','product_description_lenght','product_photos_qty','product_category_name'],axis=1)

In [34]:
order_product=order_it.merge(product, on='product_id')
order_product=order_product.dropna()
order_product.info()

<class 'pandas.DataFrame'>
Index: 112632 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   order_id           112632 non-null  str    
 1   order_item_id      112632 non-null  int64  
 2   product_id         112632 non-null  str    
 3   product_weight_g   112632 non-null  float64
 4   product_length_cm  112632 non-null  float64
 5   product_height_cm  112632 non-null  float64
 6   product_width_cm   112632 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 6.9 MB


In [39]:
order_product=order_product.drop(['product_id','order_item_id'],axis=1)

In [57]:
order_pp=order_product.merge(df_order_payments, on='order_id')
order_pp

,order_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,650.0,28.0,9.0,14.0,1,credit_card,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,30000.0,50.0,30.0,40.0,1,credit_card,3,259.83
2,000229ec398224ef6ca0657da4fc703e,3050.0,33.0,13.0,33.0,1,credit_card,5,216.87
3,00024acbcdf0a6daa1e931b038114c75,200.0,16.0,10.0,15.0,1,credit_card,2,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,35.0,40.0,30.0,1,credit_card,3,218.04
...,...,...,...,...,...,...,...,...,...
117576,fffc94f6ce00a00581880bf54a75a037,10150.0,89.0,15.0,40.0,1,boleto,1,343.40
117577,fffcd46ef2263f404302a634eb57f7eb,8950.0,45.0,26.0,38.0,1,boleto,1,386.53
117578,fffce4705a9662cd70adb13d4a31832d,967.0,21.0,24.0,19.0,1,credit_card,3,116.85
117579,fffe18544ffabc95dfada21779c9644f,100.0,20.0,20.0,20.0,1,credit_card,3,64.71


In [ ]:
df_order_dataset=df_order_dataset[df_order_dataset['order_status']=='delivered']#filtarer solo i consegnati


In [ ]:
df_order_dataset=df_order_dataset.drop(['order_status','customer_id'],axis=1)

In [62]:
df=order_pp.merge(df_order_dataset, on='order_id')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115015 entries, 0 to 115014
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       115015 non-null  str    
 1   product_weight_g               115015 non-null  float64
 2   product_length_cm              115015 non-null  float64
 3   product_height_cm              115015 non-null  float64
 4   product_width_cm               115015 non-null  float64
 5   payment_sequential             115015 non-null  int64  
 6   payment_type                   115015 non-null  str    
 7   payment_installments           115015 non-null  int64  
 8   payment_value                  115015 non-null  float64
 9   order_purchase_timestamp       115015 non-null  str    
 10  order_approved_at              115000 non-null  str    
 11  order_delivered_carrier_date   115013 non-null  str    
 12  order_delivered_customer_date  115007 non

In [64]:

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')  # 'coerce' trasforma valori non validi in NaN

In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115015 entries, 0 to 115014
Data columns (total 15 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       115015 non-null  str           
 1   product_weight_g               115015 non-null  float64       
 2   product_length_cm              115015 non-null  float64       
 3   product_height_cm              115015 non-null  float64       
 4   product_width_cm               115015 non-null  float64       
 5   payment_sequential             115015 non-null  int64         
 6   payment_type                   115015 non-null  str           
 7   payment_installments           115015 non-null  int64         
 8   payment_value                  115015 non-null  float64       
 9   order_purchase_timestamp       115015 non-null  datetime64[us]
 10  order_approved_at              115000 non-null  datetime64[us]
 11  order_deliv

In [68]:
df['actual_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

In [70]:
# tempo di approvazione
df['approval_time_days'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.days

# tempo spedizione dal corriere al cliente
df['shipping_time_days'] = (df['order_delivered_customer_date'] - df['order_delivered_carrier_date']).dt.days

# mese e giorno settimana dell'acquisto
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_weekday'] = df['order_purchase_timestamp'].dt.weekday

In [71]:
df = pd.get_dummies(df, columns=['payment_type'], drop_first=True, dtype=int)

In [73]:
df = df.dropna(subset=['actual_delivery_days', 'approval_time_days', 'shipping_time_days'])

In [75]:
df.info()

<class 'pandas.DataFrame'>
Index: 114991 entries, 0 to 115014
Data columns (total 21 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       114991 non-null  str           
 1   product_weight_g               114991 non-null  float64       
 2   product_length_cm              114991 non-null  float64       
 3   product_height_cm              114991 non-null  float64       
 4   product_width_cm               114991 non-null  float64       
 5   payment_sequential             114991 non-null  int64         
 6   payment_installments           114991 non-null  int64         
 7   payment_value                  114991 non-null  float64       
 8   order_purchase_timestamp       114991 non-null  datetime64[us]
 9   order_approved_at              114991 non-null  datetime64[us]
 10  order_delivered_carrier_date   114991 non-null  datetime64[us]
 11  order_delivered_

In [79]:
features = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'payment_sequential',
    'payment_installments',
    
] + [col for col in df.columns if col.startswith('payment_type_')]

In [80]:
y = df['actual_delivery_days']
X = df[features]

In [81]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print("R²:", model.score(X_test, y_test))

R²: 0.015378555995066212
